# TP J3 – UC-06 · Relecture des zones Silver et Gold (Parquet)

**Retail Intelligence Platform** · 18/11/2026

Ce notebook **ne produit rien** : il relit ce que les scripts `transformations/01…03` ont écrit dans HDFS, pour
vérifier que les fichiers Parquet sont présents, lisibles, avec le bon schéma et le bon nombre de lignes.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath("../transformations"))
from pyspark.sql import functions as F
import common as C
spark = C.spark_session("TP3-Relecture")
print("Spark", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/18 15:35:24 WARN Utils: Your hostname, Meddy, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/11/18 15:35:24 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/11/18 15:35:25 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.3.0


## 1. Silver : ventes magasin (partitionnées par année)

In [2]:
ventes = spark.read.parquet(f"{C.SILVER}/store_sales")
ventes.printSchema()
print("Lignes :", ventes.count())
ventes.select("ss_ticket_number", "ss_item_sk", "ss_store_sk", "sold_date", "ss_quantity", "ss_net_paid",
              "ss_net_profit", "_flag_cle_partagee", "_flag_marge_negative", "_lot", "_run_id_silver").show(5, truncate=False)

root
 |-- ss_sold_date_sk: integer (nullable = true)
 |-- ss_ticket_number: long (nullable = true)
 |-- ss_item_sk: integer (nullable = true)
 |-- ss_sold_time_sk: integer (nullable = true)
 |-- ss_customer_sk: integer (nullable = true)
 |-- ss_cdemo_sk: integer (nullable = true)
 |-- ss_hdemo_sk: integer (nullable = true)
 |-- ss_addr_sk: integer (nullable = true)
 |-- ss_store_sk: integer (nullable = true)
 |-- ss_promo_sk: integer (nullable = true)
 |-- ss_quantity: integer (nullable = true)
 |-- ss_wholesale_cost: decimal(7,2) (nullable = true)
 |-- ss_list_price: decimal(7,2) (nullable = true)
 |-- ss_sales_price: decimal(7,2) (nullable = true)
 |-- ss_ext_discount_amt: decimal(7,2) (nullable = true)
 |-- ss_ext_sales_price: decimal(7,2) (nullable = true)
 |-- ss_ext_wholesale_cost: decimal(7,2) (nullable = true)
 |-- ss_ext_list_price: decimal(7,2) (nullable = true)
 |-- ss_ext_tax: decimal(7,2) (nullable = true)
 |-- ss_coupon_amt: decimal(7,2) (nullable = true)
 |-- ss_net_paid

Lignes : 667579


+----------------+----------+-----------+----------+-----------+-----------+-------------+------------------+--------------------+-------+---------------+
|ss_ticket_number|ss_item_sk|ss_store_sk|sold_date |ss_quantity|ss_net_paid|ss_net_profit|_flag_cle_partagee|_flag_marge_negative|_lot   |_run_id_silver |
+----------------+----------+-----------+----------+-----------+-----------+-------------+------------------+--------------------+-------+---------------+
|53979           |8617      |11         |2004-01-01|72         |656.64     |126.72       |false             |false               |initial|20261118_152733|
|54000           |3624      |8          |2004-01-01|7          |368.97     |-269.36      |false             |true                |initial|20261118_152733|
|53991           |2007      |1          |2004-01-01|36         |1314.36    |-2104.56     |false             |true                |initial|20261118_152733|
|53991           |2571      |1          |2004-01-01|89         |262.53

**Lecture sélective.** Grâce au partitionnement `sold_year=…`, un filtre sur l'année ne lit qu'un dossier
(*partition pruning*) ; grâce au format colonne, seules les colonnes demandées sont lues (*ReadSchema*).

In [3]:
q = ventes.filter("sold_year = 2004").select("ss_store_sk", "ss_net_paid")
plan = q._jdf.queryExecution().executedPlan().toString()
print([l.strip() for l in plan.splitlines() if "FileScan" in l][0][:400])
print("Lignes 2004 :", q.count())

+- FileScan parquet [ss_store_sk#8,ss_net_paid#20,sold_year#32] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[hdfs://localhost:9000/retail-lake/silver/tpcxbb/store_sales], PartitionFilters: [isnotnull(sold_year#32), (sold_year#32 = 2004)], PushedFilters: [], ReadSchema: struct<ss_store_sk:int,ss_net_paid:decimal(7,2)>


Lignes 2004 : 136014


## 2. Silver : dimensions

In [4]:
for t in ["item", "store", "date_dim"]:
    df = spark.read.parquet(f"{C.SILVER}/{t}")
    print(f"{t:9s} {df.count():>7,d} lignes, {len(df.columns)} colonnes")
spark.read.parquet(f"{C.SILVER}/store").select("s_store_sk", "s_store_name", "s_closed_date",
    "s_closed_date_statut", "s_closed_date_fiable").orderBy("s_store_sk").show(truncate=False)

item       17,820 lignes, 27 colonnes


store          12 lignes, 36 colonnes


date_dim  109,573 lignes, 32 colonnes


+----------+------------+-------------+---------------------------+--------------------+
|s_store_sk|s_store_name|s_closed_date|s_closed_date_statut       |s_closed_date_fiable|
+----------+------------+-------------+---------------------------+--------------------+
|0         |Forest      |2095-01-16   |non_verifiable_apres_ventes|false               |
|1         |Hillside    |1952-01-03   |incoherente_avant_ventes   |false               |
|2         |Sherwood    |1924-05-15   |incoherente_avant_ventes   |false               |
|3         |Pioneer     |2091-05-21   |non_verifiable_apres_ventes|false               |
|4         |Elmwood     |2117-12-26   |non_verifiable_apres_ventes|false               |
|5         |Egypt       |2141-04-26   |non_verifiable_apres_ventes|false               |
|6         |Bloomfield  |1943-02-26   |incoherente_avant_ventes   |false               |
|7         |Buffalo     |2095-08-12   |non_verifiable_apres_ventes|false               |
|8         |Williamst

## 3. Gold G1 : ventes mensuelles par catégorie (Q1, Q3)

In [5]:
g1 = spark.read.parquet(f"{C.GOLD}/ventes_mensuelles_categorie")
g1.printSchema()
print("Lignes :", g1.count(), "| grain : (annee, mois, categorie)")
g1.filter("annee = 2004 AND mois = 12").select("annee", "mois", "categorie", "ca_net_paid", "quantite", "nb_tickets",
    "taux_remise_pct", "mois_complet").orderBy(F.desc("ca_net_paid")).show(truncate=False)

root
 |-- mois: integer (nullable = true)
 |-- categorie: string (nullable = true)
 |-- ca_net_paid: decimal(17,2) (nullable = true)
 |-- montant_prix_catalogue: decimal(17,2) (nullable = true)
 |-- remise: decimal(17,2) (nullable = true)
 |-- coupons: decimal(17,2) (nullable = true)
 |-- marge_nette: decimal(17,2) (nullable = true)
 |-- quantite: long (nullable = true)
 |-- nb_lignes: long (nullable = true)
 |-- nb_tickets: long (nullable = true)
 |-- taux_remise_pct: decimal(24,2) (nullable = true)
 |-- mois_complet: boolean (nullable = true)
 |-- _source: string (nullable = true)
 |-- _run_id_gold: string (nullable = true)
 |-- annee: integer (nullable = true)



Lignes : 540 | grain : (annee, mois, categorie)


+-----+----+------------------------+-----------+--------+----------+---------------+------------+
|annee|mois|categorie               |ca_net_paid|quantite|nb_tickets|taux_remise_pct|mois_complet|
+-----+----+------------------------+-----------+--------+----------+---------------+------------+
|2004 |12  |Music                   |9305895.54 |270109  |1412      |49.69          |true        |
|2004 |12  |Books                   |5287727.61 |154850  |1248      |49.79          |true        |
|2004 |12  |Movies & TV             |1312717.42 |39030   |589       |50.43          |true        |
|2004 |12  |Clothing & Accessories  |1023626.87 |27644   |458       |48.06          |true        |
|2004 |12  |Electronics             |955568.71  |28591   |457       |51.32          |true        |
|2004 |12  |Sports & Outdoors       |781946.22  |21235   |371       |46.59          |true        |
|2004 |12  |Home & Kitchen          |749785.78  |22386   |383       |48.21          |true        |
|2004 |12 

In [6]:
# évolution annuelle du CA (Q1) : 2005 est signalée incomplète
(g1.groupBy("annee").agg(F.round(F.sum("ca_net_paid") / 1e6, 2).alias("ca_M"),
                         F.min(F.col("mois_complet").cast("int")).alias("annee_complete_si_1"))
   .orderBy("annee").show())

+-----+------+-------------------+
|annee|  ca_M|annee_complete_si_1|
+-----+------+-------------------+
| 2001| 232.5|                  1|
| 2002|230.95|                  1|
| 2003|233.58|                  1|
| 2004|234.34|                  1|
| 2005|219.16|                  0|
+-----+------+-------------------+



## 4. Gold G2 : performance annuelle des magasins (Q5)

In [7]:
g2 = spark.read.parquet(f"{C.GOLD}/performance_magasin_annuelle")
print("Lignes :", g2.count(), "| grain : (annee, magasin)")
g2.filter("annee = 2004").select("rang_ca_annee", "s_store_name", "s_state", "ca_net_paid", "nb_tickets",
    "panier_moyen", "part_ca_annee_pct", "ecart_moyenne_pct").orderBy("rang_ca_annee").show(truncate=False)

Lignes : 60 | grain : (annee, magasin)


+-------------+------------+-------+-----------+----------+------------+-----------------+-----------------+
|rang_ca_annee|s_store_name|s_state|ca_net_paid|nb_tickets|panier_moyen|part_ca_annee_pct|ecart_moyenne_pct|
+-------------+------------+-------+-----------+----------+------------+-----------------+-----------------+
|1            |Buffalo     |KS     |20748576.85|1569      |13224.08    |8.85             |6.25             |
|2            |Williamstown|TX     |20717130.74|1530      |13540.61    |8.84             |6.09             |
|3            |Pioneer     |TX     |20282799.41|1564      |12968.54    |8.66             |3.86             |
|4            |Bedford     |KY     |19813076.58|1508      |13138.64    |8.45             |1.46             |
|5            |Germantown  |WV     |19732578.43|1520      |12981.96    |8.42             |1.04             |
|6            |Hillside    |TN     |19568501.11|1491      |13124.41    |8.35             |0.20             |
|7            |Elmw

## 5. Rapport qualité (zone quality, historique des exécutions)

In [8]:
rq = spark.read.parquet(f"{C.QUALITY}/rapport_controles")
rq.groupBy("run_id", "statut").count().orderBy("run_id", "statut").show()
spark.stop()

+---------------+------+-----+
|         run_id|statut|count|
+---------------+------+-----+
|20261118_152234|ALERTE|    7|
|20261118_152234|    OK|   24|
|20261118_153056|ALERTE|    7|
|20261118_153056|    OK|   24|
+---------------+------+-----+

